# MemoryWatch — Deliverable 2, Notebook 2: Preprocessing

Transforms the raw UNSW-NB15 columns into a model-ready feature matrix:

- Drop identifier / label columns (`id`, `attack_cat`, `label`) from the feature set
- Label-encode the three categorical columns (`proto`, `service`, `state`)
- Min-Max scale all numeric columns to `[0, 1]`
- **Fit** the encoding vocabulary and the min/max scaling bounds on the **training set only**, then reuse them (no refitting) on the test set — this avoids test-set leakage

Logic lives in `utils.preprocess()` so both this notebook and any future re-runs use the exact same transformation. Processed arrays are saved to `memorywatch_processed.npz` for Notebook 3.

In [1]:
import sys
sys.path.insert(0, '.')
import numpy as np
import utils

train, test = utils.load_raw()
X_tr, y_tr, at_tr, vocab, lo, hi = utils.preprocess(train, fit=True)
X_te, y_te, at_te, *_            = utils.preprocess(test, vocab=vocab, lo=lo, hi=hi, fit=False)

print(f"Feature shape — train {X_tr.shape}, test {X_te.shape}")
print(f"Categorical vocab sizes: " + ', '.join(f'{k}={len(v)}' for k,v in vocab.items()))


Feature shape — train (175341, 42), test (82332, 42)
Categorical vocab sizes: proto=133, service=13, state=9


In [2]:
X_norm = X_tr[y_tr == 0]
print(f"Normal (train)  : {len(X_norm):,}")
print(f"Attack (train)  : {(y_tr==1).sum():,}")
print(f"Normal (test)   : {(y_te==0).sum():,}")
print(f"Attack (test)   : {(y_te==1).sum():,}")
print(f"Value range check — min={X_tr.min():.3f}, max={X_tr.max():.3f} (should be [0,1])")


Normal (train)  : 56,000
Attack (train)  : 119,341
Normal (test)   : 37,000
Attack (test)   : 45,332
Value range check — min=0.000, max=1.000 (should be [0,1])


In [3]:
np.savez(utils.OUT_DIR + 'memorywatch_processed.npz',
         X_tr=X_tr, y_tr=y_tr, at_tr=at_tr,
         X_te=X_te, y_te=y_te, at_te=at_te)
print('Saved memorywatch_processed.npz')
print(f"  X_tr {X_tr.shape}, X_te {X_te.shape}")


Saved memorywatch_processed.npz
  X_tr (175341, 42), X_te (82332, 42)
